# 068 — Síntesis de voz y clonación responsable

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Pipeline TTS:** texto → **normalización** (números, siglas, fechas) + **G2P** (letras →
fonemas) → **modelo acústico** (Tacotron 2: seq2seq con atención que predice el log-mel de
80 bandas, hop ~12,5 ms, más un *stop token*) → **vocoder** (mel → onda, porque el mel
descarta la fase).

**Vocoders:** WaveNet (2016) genera muestra a muestra con convoluciones causales dilatadas
(`p(x) = ∏ p(xₜ|x₁…xₜ₋₁)`): calidad casi humana, pero 16 000 pasos por segundo de audio.
Los vocoders paralelos (HiFi-GAN) generan toda la onda en una pasada → tiempo real.

**Clonación:** un *speaker encoder* comprime segundos de voz en un d-vector que condiciona
el modelo acústico (SV2TTS). **Responsabilidad:** consentimiento explícito y documentado,
watermarking imperceptible (degradable por re-grabación/compresión) y detección de audio
sintético. **Evaluación:** MOS (1-5, subjetivo, reportar media *y* desvío) y WER de un ASR
sobre el audio sintético como proxy de inteligibilidad.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (a) Campo = `1 + 2·(1+2+…+512) = 1 + 2·1023 = 2047` muestras →
`2047/16000 ≈ 128 ms` de contexto. (b) Autoregresivo: `1.5 · 16 000 = 24 000` pasos uno
tras otro (cada muestra espera a la anterior). Paralelo: una sola pasada de red para las
24 000 muestras — esa es la diferencia entre demo y asistente en tiempo real.

**Ejercicio 2.** `4 s / 12,5 ms = 320` tramas; la salida es una matriz `320 × 80` (más el
stop token por trama). El vocoder convierte después esas 320 tramas en 64 000 muestras.

**Ejercicio 3.** Ambos tienen media 4,0. Desvíos: A → 0,63; B → 1,55. Para un lector de
noticias (uso continuo) conviene A: B suena excelente casi siempre pero falla fuerte a
veces (el 1), y los fallos graves ocasionales destruyen la confianza más que una calidad
media uniforme. La media idéntica esconde distribuciones muy distintas.

**Ejercicio 4.** Implementación debajo: imprime 2047, (4.0, 0.632…), (4.0, 1.549…).


In [ ]:
result = run_lab("generation", seed=68)
assert result["kind"] == "generation"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicios 1 y 3 — verificados con código
def receptive_field(dilaciones, pilas=1):
    return 1 + pilas * sum(dilaciones)

def mos(ratings):
    n = len(ratings)
    media = sum(ratings) / n
    var = sum((r - media) ** 2 for r in ratings) / n
    return media, var ** 0.5

dil = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
rf = receptive_field(dil, 2)
print("campo receptivo:", rf, "muestras =", round(rf / 16000 * 1000, 1), "ms")
print("pasos autoregresivos para 1.5 s a 16 kHz:", int(1.5 * 16000))
print("MOS A:", mos([4, 4, 5, 3, 4]))
print("MOS B:", mos([5, 5, 5, 1, 4]))


In [ ]:
# Ejercicio 2 — tramas del modelo acústico
tramas = int(4 / 0.0125)
print("tramas:", tramas, "| salida:", (tramas, 80), "| muestras finales:", 4 * 16000)


## Reflexión

1. Una familia pide clonar la voz de un pariente fallecido para un homenaje. ¿Quién puede
   consentir en ese caso, y qué límites de alcance y revocación pondrías por escrito?
2. El watermark de tu TTS sobrevive a la compresión MP3 pero no a la re-grabación con un
   micrófono. ¿Sigue siendo útil? ¿Dentro de qué estrategia de defensa más amplia?
3. Tu TTS lee recetas médicas en voz alta (clase 072). ¿Qué error del frontend de
   normalización sería el más peligroso ("500 mg", "c/8 h") y cómo lo detectarías antes de
   desplegar?
